# Grammar Scoring Engine for Spoken Audio

**Task:** given a 45–60 s `.wav` recording, predict a continuous grammar score in [0, 5] (MOS Likert rubric).
**Metrics:** RMSE (lower is better) and Pearson correlation (higher is better).

## Report summary

**What the first experiments showed.** A baseline that predicts the mean has RMSE ≈ 1.24. In cross-validation the *acoustic* Whisper-encoder embedding was by far the strongest single view (Pearson ≈ 0.89), the hand-crafted text features were weaker (≈ 0.77) and the sentence-embedding of the transcript weakest (≈ 0.65). Spoken grammar proficiency is therefore expressed strongly in *how* a person speaks (fluency, rhythm, hesitation, pronunciation), which the encoder captures well. The final pipeline invests accordingly:

1. **Acoustic view (main signal).** Mean- *and* standard-deviation-pooled hidden states from **every layer** of the Whisper encoder (bigger model, 2 × 30 s windows). Intermediate layers often carry more useful information than the last one, so we probe each layer by cross-validation and keep the best few.
2. **Text view.** Whisper ASR transcript → surface statistics, ASR confidence, **GPT-2 perplexity** and **CoLA grammatical-acceptability** scores (RoBERTa fine-tuned to judge whether a sentence is grammatical), plus a MiniLM sentence embedding.
3. **Models.** Ridge (with built-in alpha selection), SVR with *nested* hyper-parameter search, gradient boosting; one family per feature view and one on everything concatenated. Out-of-fold predictions are blended by a non-negative linear stacker; outputs are clipped to [0, 5].

**Evaluation.** 5-fold cross-validated RMSE / Pearson (honest estimate; hyper-parameter search is nested inside each fold) plus the compulsory **RMSE on the full training data** after the final fit.

> **Setup:** runs locally (VS Code / Jupyter) and uses the Apple GPU (`mps`) / CUDA when available. Needs `ffmpeg` (`brew install ffmpeg`). Heavy steps are cached in `./working` so re-runs are fast.

## 1. Setup & configuration

In [ ]:
%pip -q install openai-whisper sentence-transformers librosa soundfile scikit-learn scipy seaborn

In [ ]:
import os, re, glob, warnings, pickle, random, copy
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # must be set BEFORE importing torch (Mac GPU fallback)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

# ---------------- CONFIG (local machine) ----------------
DATA_ROOT       = "/Users/apple/Downloads/Dataset_Final"   # folder with train.csv, test.csv, train/, test/
TRAIN_AUDIO_DIR = os.path.join(DATA_ROOT, "train")
TEST_AUDIO_DIR  = os.path.join(DATA_ROOT, "test")

WORK_DIR   = "./working"     # caches + submission.csv are saved here
ASR_SIZE   = "base"          # Whisper size used for TRANSCRIPTS (text features are the weaker view -> "base" is enough)
ENC_SIZE   = "small"         # Whisper size used for ACOUSTIC EMBEDDINGS (the strongest view). Try "medium" for more accuracy
QUICK_TEST = False           # True -> 30 train / 10 test files, just to check that everything runs
TAG = "_quick" if QUICK_TEST else ""
ASR_CACHE  = os.path.join(WORK_DIR, f"asr_{ASR_SIZE}{TAG}.pkl")
EMB_CACHE  = os.path.join(WORK_DIR, f"emb_{ENC_SIZE}{TAG}.pkl")
LEGACY_ASR = os.path.join(WORK_DIR, f"features_{ASR_SIZE}.pkl")   # cache written by the previous notebook version (reused if present)
os.makedirs(WORK_DIR, exist_ok=True)

def hf_load(fn, *a, **k):
    """Load a Hugging Face model from the local cache first (no network retries), download only if needed."""
    try:    return fn(*a, local_files_only=True, **k)
    except Exception: return fn(*a, **k)

In [ ]:
train_df   = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
test_df    = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))

if QUICK_TEST:
    train_df, test_df = train_df.head(30).copy(), test_df.head(10).copy()

train_df["path"] = train_df.filename.map(lambda f: os.path.join(TRAIN_AUDIO_DIR, f))
test_df["path"]  = test_df.filename.map(lambda f: os.path.join(TEST_AUDIO_DIR, f))

n_miss_tr = (~train_df.path.map(os.path.exists)).sum()
n_miss_te = (~test_df.path.map(os.path.exists)).sum()
print("train:", train_df.shape, "| missing audio:", n_miss_tr)
print("test :", test_df.shape,  "| missing audio:", n_miss_te)
assert n_miss_tr == 0 and n_miss_te == 0, "Audio files not found - check TRAIN_AUDIO_DIR / TEST_AUDIO_DIR"
y = train_df.label.values.astype(float)
train_df.head()

## 2. Exploratory analysis

In [ ]:
import soundfile as sf
durs = [sf.info(p).duration for p in train_df.path[:150]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=train_df.label, ax=ax[0], color="steelblue")
ax[0].set_title("Training label distribution"); ax[0].set_xlabel("Grammar score")
ax[1].hist(durs, bins=20, color="darkorange"); ax[1].set_title("Audio duration (s), first 150 files")
plt.tight_layout(); plt.show()
print(train_df.label.describe())
print("Sample rate:", sf.info(train_df.path[0]).samplerate)

**Observations.** Labels are concentrated around 2–5; a block of `0` scores (likely empty / unintelligible recordings) and almost no 1–1.5. Predicting the mean (~3.3) gives RMSE ≈ 1.24, the baseline to beat. Clips are 45–60 s, so a single 30 s Whisper window would discard half of each recording – we process two windows.

## 3. Feature extraction

### 3.1 Transcripts (Whisper ASR)
Transcripts + ASR confidence are cached. If the cache written by the previous notebook version (`features_base.pkl`) exists it is reused, which saves ~20 minutes.

In [ ]:
import whisper

def transcribe(asr, audio):
    res = asr.transcribe(audio, language="en", fp16=(DEVICE == "cuda"),
                         temperature=0.0, condition_on_previous_text=False)
    segs = res["segments"]
    return dict(
        text=res["text"].strip(),
        avg_logprob=np.mean([s["avg_logprob"] for s in segs]) if segs else -5.0,
        min_logprob=np.min([s["avg_logprob"] for s in segs]) if segs else -5.0,
        no_speech=np.mean([s["no_speech_prob"] for s in segs]) if segs else 1.0,
        compression=np.mean([s["compression_ratio"] for s in segs]) if segs else 0.0,
        n_segments=len(segs),
        duration=len(audio) / 16000,
    )

def load_whisper(size, device):
    m = whisper.load_model(size, device="cpu")
    if device == "mps":
        m._buffers["alignment_heads"] = None   # sparse buffer breaks .to("mps"); only needed for word timestamps
        m = m.to(device)
    return m

def run_asr(df):
    return pd.DataFrame([transcribe(asr, whisper.load_audio(p)) for p in tqdm(df.path, desc=f"ASR ({ASR_SIZE})")])

if os.path.exists(ASR_CACHE):
    d = pickle.load(open(ASR_CACHE, "rb")); tr_asr, te_asr = d["tr_asr"], d["te_asr"]
    print("Loaded ASR cache:", ASR_CACHE)
elif (not QUICK_TEST) and os.path.exists(LEGACY_ASR):
    d = pickle.load(open(LEGACY_ASR, "rb")); tr_asr, te_asr = d["tr_asr"], d["te_asr"]
    assert len(tr_asr) == len(train_df) and len(te_asr) == len(test_df)
    pickle.dump(dict(tr_asr=tr_asr, te_asr=te_asr), open(ASR_CACHE, "wb"))
    print("Reused transcripts from the previous run:", LEGACY_ASR)
else:
    asr, asr_dev = None, ("cpu" if DEVICE == "mps" else DEVICE)
    if DEVICE == "mps":                                  # try the Mac GPU, fall back to CPU if it misbehaves
        try:
            asr = load_whisper(ASR_SIZE, "mps")
            probe = asr.transcribe(whisper.load_audio(train_df.path.iloc[0]), language="en", fp16=False)["text"]
            assert len(probe.strip()) > 20, "empty transcript on MPS"
            asr_dev = "mps"
        except Exception as e:
            print("Transcription on MPS failed -> CPU. Reason:", repr(e)[:150]); asr = None
    if asr is None:
        asr = whisper.load_model(ASR_SIZE, device=asr_dev)
    print("ASR running on:", asr_dev)
    tr_asr, te_asr = run_asr(train_df), run_asr(test_df)
    pickle.dump(dict(tr_asr=tr_asr, te_asr=te_asr), open(ASR_CACHE, "wb"))
    del asr
print(tr_asr.shape, te_asr.shape)
tr_asr.head(3)

### 3.2 Acoustic embeddings from *every* Whisper-encoder layer
For each recording we pass up to two 30 s windows through the encoder (on the GPU) and, **for every layer**, store the **mean** and **standard deviation** over time of the hidden states (only over frames that contain audio, not padding). Mean captures the average "speaking style"; std captures variability (hesitations, pacing changes).

In [ ]:
def load_encoder(size):
    m = whisper.load_model(size, device="cpu")           # load on CPU, then move ONLY the encoder to the GPU
    enc = m.encoder.to(DEVICE).eval()
    return enc, m.dims

enc, dims = load_encoder(ENC_SIZE)
N_LAYERS, D_MODEL = len(enc.blocks), dims.n_audio_state
print(f"Whisper-{ENC_SIZE} encoder: {N_LAYERS} layers x {D_MODEL} dims, on {DEVICE}")

_store = []
for blk in enc.blocks:                                    # forward hooks collect every layer's output
    blk.register_forward_hook(lambda mod, inp, out: _store.append(out if torch.is_tensor(out) else out[0]))

@torch.no_grad()
def encoder_stats(audio):
    """-> (mean, std), each (N_LAYERS, D_MODEL): per-layer pooled hidden states over valid frames of up to 2 windows."""
    win = 30 * 16000
    frames = [[] for _ in range(N_LAYERS)]
    for start in range(0, max(len(audio), 1), win)[:2]:
        chunk = audio[start:start + win]
        if len(chunk) < 2 * 16000:                        # ignore tiny tails
            continue
        n_valid = min(int(np.ceil(len(chunk) / 16000 * 50)), 1500)   # encoder frames: 50 / second
        mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(chunk)).to(DEVICE)
        if DEVICE == "cuda": mel = mel.half()
        _store.clear()
        enc(mel.unsqueeze(0))
        for l, h in enumerate(_store[:N_LAYERS]):
            frames[l].append(h[0, :n_valid].float().cpu())
    if not frames[0]:
        z = np.zeros((N_LAYERS, D_MODEL), np.float32); return z, z
    allf = [torch.cat(f, 0) for f in frames]
    return (np.stack([f.mean(0).numpy() for f in allf]).astype(np.float32),
            np.stack([f.std(0).numpy()  for f in allf]).astype(np.float32))

def extract_emb(df):
    ms, ss = [], []
    for p in tqdm(df.path, desc=f"Whisper-{ENC_SIZE} encoder"):
        m, s = encoder_stats(whisper.load_audio(p)); ms.append(m); ss.append(s)
    return np.stack(ms), np.stack(ss)               # (n, L, d)

if os.path.exists(EMB_CACHE):
    d = pickle.load(open(EMB_CACHE, "rb")); print("Loaded embedding cache:", EMB_CACHE)
else:
    tr_m, tr_s = extract_emb(train_df); te_m, te_s = extract_emb(test_df)
    d = dict(tr_m=tr_m, tr_s=tr_s, te_m=te_m, te_s=te_s)
    pickle.dump(d, open(EMB_CACHE, "wb"))
tr_m, tr_s, te_m, te_s = d["tr_m"], d["tr_s"], d["te_m"], d["te_s"]
print("train mean/std:", tr_m.shape, tr_s.shape, "| test:", te_m.shape)
del enc; import gc; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

### 3.3 Cross-validation setup and layer-wise probe
Which encoder layer carries the most grammar information? We fit a Ridge regressor on each layer's mean-pooled embedding and compare 5-fold CV RMSE. The same folds are reused for every model later.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
strat = np.clip(np.round(y), 1, 5).astype(int)           # merge the rare 0/1 scores into one stratum
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), strat))

def cv_predict(make, X):
    p = np.zeros(len(y))
    for tr_i, va_i in folds:
        p[va_i] = make().fit(X[tr_i], y[tr_i]).predict(X[va_i])
    return np.clip(p, 0, 5)

layer_rmse = []
for l in range(N_LAYERS):
    p = cv_predict(lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 4.5, 10))), tr_m[:, l])
    layer_rmse.append(rmse(y, p))
layer_rmse = np.array(layer_rmse)
TOP_LAYERS = sorted(np.argsort(layer_rmse)[:3].tolist())
BEST_LAYER = int(np.argmin(layer_rmse))

plt.figure(figsize=(8, 4)); plt.plot(range(N_LAYERS), layer_rmse, "o-")
plt.scatter(TOP_LAYERS, layer_rmse[TOP_LAYERS], color="red", zorder=3, label="selected layers")
plt.xlabel("Whisper encoder layer"); plt.ylabel("CV RMSE (Ridge)"); plt.title("Layer-wise probe: where is grammar information?")
plt.legend(); plt.grid(alpha=.3); plt.show()
print("per-layer CV RMSE:", np.round(layer_rmse, 3))
print("selected layers:", TOP_LAYERS, "| best single layer:", BEST_LAYER)

### 3.4 Text features: surface statistics + GPT-2 perplexity
* **Perplexity** of the transcript under GPT-2 is a proxy for grammatical naturalness (overall and per sentence: mean / median / max).
* Surface features: word count, speaking rate, type-token ratio, sentence-length statistics, filler-word rate, immediate word repetitions (repairs / false starts), long-word share, and subordinating-conjunction rate (proxy for complex grammar).

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
tok = hf_load(GPT2TokenizerFast.from_pretrained, "gpt2")
lm  = hf_load(GPT2LMHeadModel.from_pretrained, "gpt2").to(DEVICE).eval()

@torch.no_grad()
def ppl(text, max_len=256):
    if len(text.split()) < 3: return np.nan
    ids = tok(text, return_tensors="pt", truncation=True, max_length=max_len).input_ids.to(DEVICE)
    ids = torch.cat([torch.tensor([[tok.bos_token_id]], device=DEVICE), ids], 1)
    loss = lm(ids, labels=ids).loss.item()
    return float(np.exp(min(loss, 15)))

FILLERS = {"um", "uh", "er", "erm", "hmm", "ah", "like", "actually", "basically"}
SUBORD  = {"because", "although", "though", "while", "whereas", "since", "unless", "if", "when", "which", "that", "who", "whose", "whom", "however", "therefore"}

def text_features(text, duration):
    words = re.findall(r"[A-Za-z']+", text.lower())
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip().split()) > 0]
    n = max(len(words), 1)
    sent_lens = [len(s.split()) for s in sents] or [0]
    sent_ppl = [ppl(s) for s in sents if len(s.split()) >= 4]
    sent_ppl = [x for x in sent_ppl if not np.isnan(x)] or [np.nan]
    return dict(
        n_words=len(words), wpm=len(words) / max(duration, 1) * 60,
        ttr=len(set(words)) / n, n_sents=len(sents),
        mean_sent_len=np.mean(sent_lens), max_sent_len=np.max(sent_lens), std_sent_len=np.std(sent_lens),
        short_sent_frac=np.mean([l <= 3 for l in sent_lens]),
        filler_rate=sum(w in FILLERS for w in words) / n,
        repeat_rate=sum(a == b for a, b in zip(words, words[1:])) / n,
        long_word_frac=np.mean([len(w) >= 7 for w in words]) if words else 0,
        subord_rate=sum(w in SUBORD for w in words) / n,
        ppl_all=ppl(text, 400), ppl_mean=np.nanmean(sent_ppl), ppl_med=np.nanmedian(sent_ppl), ppl_max=np.nanmax(sent_ppl),
    )

def build_hand_features(asr_df):
    tf = pd.DataFrame([text_features(t, d) for t, d in tqdm(zip(asr_df.text, asr_df.duration), total=len(asr_df), desc="text feats")])
    out = pd.concat([asr_df.drop(columns=["text"]).reset_index(drop=True), tf], axis=1)
    for c in ["ppl_all", "ppl_mean", "ppl_med", "ppl_max"]:
        out[c] = np.log1p(out[c])            # perplexity is heavy-tailed -> log
    out["compression"] = out["compression"].clip(upper=5)          # a few hallucination loops give extreme values
    out["max_sent_len"] = out["max_sent_len"].clip(upper=100)
    out["mean_sent_len"] = out["mean_sent_len"].clip(upper=60)
    return out

X_hand_tr = build_hand_features(tr_asr)
X_hand_te = build_hand_features(te_asr)
med = X_hand_tr.median()
X_hand_tr, X_hand_te = X_hand_tr.fillna(med), X_hand_te.fillna(med)   # empty transcripts -> median
X_hand_tr.describe().T.head(25)

### 3.5 Grammar-correctness features (CoLA)
A RoBERTa classifier fine-tuned on **CoLA** (Corpus of Linguistic Acceptability) scores every transcript sentence for grammatical acceptability. Per recording we keep the mean probability, the worst sentence, and the fraction of sentences judged acceptable. A sanity check on two example sentences decides which output index means "acceptable". Runs on the GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

cola_name = "textattack/roberta-base-CoLA"
cola_tok = hf_load(AutoTokenizer.from_pretrained, cola_name)
cola = hf_load(AutoModelForSequenceClassification.from_pretrained, cola_name).to(DEVICE).eval()

@torch.no_grad()
def cola_probs(sents, idx):
    enc_ = cola_tok(sents, return_tensors="pt", padding=True, truncation=True, max_length=64).to(DEVICE)
    return cola(**enc_).logits.softmax(-1)[:, idx].cpu().numpy()

good, bad = "She went to the market and bought some fresh vegetables.", "She go market buy vegetable yesterday not."
ACCEPT_IDX = 1 if cola_probs([good], 1)[0] > cola_probs([bad], 1)[0] else 0
print("acceptable-sentence index:", ACCEPT_IDX, "| good:", cola_probs([good], ACCEPT_IDX)[0].round(3), "| bad:", cola_probs([bad], ACCEPT_IDX)[0].round(3))

def cola_feats(text):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 3]
    if not sents:
        return dict(cola_mean=np.nan, cola_min=np.nan, cola_frac=np.nan)
    p = np.concatenate([cola_probs(sents[i:i + 32], ACCEPT_IDX) for i in range(0, len(sents), 32)])
    return dict(cola_mean=p.mean(), cola_min=p.min(), cola_frac=(p > 0.5).mean())

cf_tr = pd.DataFrame([cola_feats(t) for t in tqdm(tr_asr.text, desc="CoLA train")])
cf_te = pd.DataFrame([cola_feats(t) for t in tqdm(te_asr.text, desc="CoLA test")])

X_hand_tr = pd.concat([X_hand_tr.reset_index(drop=True), cf_tr], axis=1)
X_hand_te = pd.concat([X_hand_te.reset_index(drop=True), cf_te], axis=1)
med = X_hand_tr.median()
X_hand_tr, X_hand_te = X_hand_tr.fillna(med), X_hand_te.fillna(med)
print("hand-crafted feature matrix:", X_hand_tr.shape)

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st_model = hf_load(SentenceTransformer, "sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    X_txt_tr = st_model.encode(tr_asr.text.tolist(), batch_size=32, show_progress_bar=True)
    X_txt_te = st_model.encode(te_asr.text.tolist(), batch_size=32, show_progress_bar=True)
    print("Using MiniLM embeddings")
except Exception as e:                                   # e.g. no internet -> local TF-IDF + SVD fallback
    print("MiniLM unavailable, using TF-IDF + SVD instead:", repr(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
    A = tfidf.fit_transform(tr_asr.text)
    svd = TruncatedSVD(min(128, A.shape[1] - 1), random_state=SEED).fit(A)
    X_txt_tr, X_txt_te = svd.transform(A), svd.transform(tfidf.transform(te_asr.text))

# acoustic feature blocks built from the encoder statistics
X_aud_tr  = np.hstack([tr_m[:, TOP_LAYERS].reshape(len(tr_m), -1), tr_s[:, BEST_LAYER]])   # mean of top layers + std of best layer
X_aud_te  = np.hstack([te_m[:, TOP_LAYERS].reshape(len(te_m), -1), te_s[:, BEST_LAYER]])
X_aud1_tr, X_aud1_te = tr_m[:, BEST_LAYER], te_m[:, BEST_LAYER]                            # best single layer (mean only)
print("acoustic:", X_aud_tr.shape, "| best layer:", X_aud1_tr.shape, "| text:", X_txt_tr.shape, "| hand:", X_hand_tr.shape)

### 3.6 Which hand-crafted features relate to the grammar score?

In [ ]:
corr = X_hand_tr.assign(label=y).corr()["label"].drop("label").sort_values()
plt.figure(figsize=(7, 7)); corr.plot.barh(color=np.where(corr > 0, "seagreen", "indianred"))
plt.title("Pearson correlation of text features with grammar score"); plt.xlabel("r"); plt.tight_layout(); plt.show()

Perplexity-type features should correlate **negatively** (more natural text → higher score) while ASR confidence, CoLA acceptability, sentence length and vocabulary richness correlate **positively**.

## 4. Modelling

Each feature view gets its own regressors, evaluated with the same stratified 5-fold CV. SVR hyper-parameters are tuned by an **inner** 3-fold grid search inside every outer fold (nested CV), so the reported CV scores stay honest. Out-of-fold (OOF) predictions are blended afterwards.

In [ ]:
NPC = lambda k: max(2, min(k, len(y) * 8 // 15 - 2))     # PCA size that also fits inside the inner CV folds (matters for the tiny QUICK_TEST subset)
ALPHAS = np.logspace(0, 4.5, 10)

def svr_grid(n_pc=128):
    pipe = make_pipeline(StandardScaler(), PCA(NPC(n_pc), random_state=SEED), SVR())
    grid = {"svr__C": [1, 3, 10, 30], "svr__epsilon": [0.1, 0.3]}
    return GridSearchCV(pipe, grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=1)

models = {
    "RidgeCV | Whisper layers":      (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS)), "aud"),
    "SVR-grid| Whisper layers":      (lambda: svr_grid(128), "aud"),
    "SVR-grid| Whisper best layer":  (lambda: svr_grid(96),  "aud1"),
    "RidgeCV | MiniLM text":         (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS)), "txt"),
    "HistGB  | text+CoLA features":  (lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.04, max_iter=300, l2_regularization=1.0, random_state=SEED), "hand"),
    "RidgeCV | text+CoLA features":  (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS)), "hand"),
    "SVR-grid| all concatenated":    (lambda: svr_grid(160), "all"),
}
TR = {"aud": X_aud_tr, "aud1": X_aud1_tr, "txt": X_txt_tr, "hand": X_hand_tr.values,
      "all": np.hstack([X_aud_tr, X_txt_tr, X_hand_tr.values])}
TE = {"aud": X_aud_te, "aud1": X_aud1_te, "txt": X_txt_te, "hand": X_hand_te.values,
      "all": np.hstack([X_aud_te, X_txt_te, X_hand_te.values])}

oof = {}
for name, (mk, blk) in models.items():
    oof[name] = cv_predict(mk, TR[blk])
    print(f"{name:30s}  CV RMSE={rmse(y, oof[name]):.3f}  Pearson={pearsonr(y, oof[name])[0]:.3f}")
print(f"{'Baseline (mean)':30s}  CV RMSE={rmse(y, np.full_like(y, y.mean())):.3f}")

### 4.1 Stacking
A non-negative linear regression learns how much to trust each base model (non-negativity keeps the blend stable with ~770 samples). Its quality is estimated with CV over the OOF matrix.

In [ ]:
names = list(models)
O = np.column_stack([oof[n] for n in names])
stack = LinearRegression(positive=True).fit(O, y)
print(dict(zip(names, np.round(stack.coef_, 3))), "intercept", round(stack.intercept_, 3))

p_stack = np.zeros(len(y))
for tr_i, va_i in folds:
    s = LinearRegression(positive=True).fit(O[tr_i], y[tr_i]); p_stack[va_i] = s.predict(O[va_i])
p_stack = np.clip(p_stack, 0, 5)
cv_rmse, cv_r = rmse(y, p_stack), pearsonr(y, p_stack)[0]
print(f"\nStacked ensemble  CV RMSE = {cv_rmse:.3f} | CV Pearson = {cv_r:.3f}")

## 5. Evaluation & visualisations (cross-validated)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
res = pd.DataFrame({"RMSE": [rmse(y, oof[n]) for n in names] + [cv_rmse]}, index=names + ["STACK"]).sort_values("RMSE")
res.RMSE.plot.barh(ax=ax[0], color="steelblue"); ax[0].axvline(rmse(y, np.full_like(y, y.mean())), color="red", ls="--", label="mean baseline")
ax[0].set_title("CV RMSE by model"); ax[0].legend()
jit = np.random.uniform(-.08, .08, len(y))
ax[1].scatter(y + jit, p_stack, alpha=.5, s=14); ax[1].plot([0, 5], [0, 5], "r--")
ax[1].set_xlabel("True score"); ax[1].set_ylabel("Predicted (OOF)"); ax[1].set_title(f"Stack: RMSE={cv_rmse:.3f}, r={cv_r:.3f}")
ax[2].hist(y - p_stack, bins=30, color="seagreen"); ax[2].set_title("Residuals (true - predicted)")
plt.tight_layout(); plt.show()

print("Mean absolute error per true label:")
print(pd.DataFrame({"label": y, "abs_err": np.abs(y - p_stack)}).groupby("label").abs_err.agg(["mean", "count"]).round(3).T)

## 6. Final fit, training-set RMSE (required) and test predictions
All base models are refit on the **full training set**, blended with the stacker, and used to predict the test set.

In [ ]:
final_models = {n: models[n][0]().fit(TR[models[n][1]], y) for n in names}
train_pred_base = np.column_stack([np.clip(final_models[n].predict(TR[models[n][1]]), 0, 5) for n in names])
train_pred = np.clip(stack.predict(train_pred_base), 0, 5)

TRAIN_RMSE = rmse(y, train_pred)
print("=" * 55)
print(f"  TRAINING RMSE (full-fit model, train data) : {TRAIN_RMSE:.4f}")
print(f"  TRAINING Pearson                           : {pearsonr(y, train_pred)[0]:.4f}")
print(f"  5-fold CV RMSE (honest generalisation)     : {cv_rmse:.4f}")
print(f"  5-fold CV Pearson                          : {cv_r:.4f}")
print("=" * 55)

test_base = np.column_stack([np.clip(final_models[n].predict(TE[models[n][1]]), 0, 5) for n in names])
test_pred = np.clip(stack.predict(test_base), 0, 5)

plt.figure(figsize=(6, 4)); sns.kdeplot(y, label="train true", fill=True); sns.kdeplot(test_pred, label="test predicted", fill=True)
plt.legend(); plt.title("Score distributions"); plt.xlim(0, 5); plt.show()

In [ ]:
# submission: same columns as sample_submission.csv, one row per file in test.csv
submission = pd.DataFrame({"filename": test_df.filename, "label": test_pred})
assert list(submission.columns) == list(sample_sub.columns)
submission.to_csv(os.path.join(WORK_DIR, "submission.csv"), index=False)
print(submission.shape); submission.head()

## 7. Discussion

**Pipeline architecture.** `wav (16 kHz)` → (a) Whisper encoder (all layers, mean + std pooled, 2 windows) → layer probe → acoustic features; (b) Whisper ASR → transcript → surface stats, GPT-2 perplexity, CoLA acceptability, MiniLM embedding. Each view → its own regressors (Ridge / SVR with nested grid search / gradient boosting) → non-negative stacker → clip to [0, 5].

**Results.** See §5–6: the *training* RMSE is (as always) lower than the cross-validated RMSE, which is the number that predicts leaderboard behaviour. Compare the CV RMSE with the mean-prediction baseline (≈ 1.24).

**Interpretation.** The layer probe shows where in the encoder grammar-relevant information sits; the acoustic view dominates because proficiency shows up in fluency and delivery; perplexity, CoLA and ASR confidence add complementary text-level evidence; stacking lowers variance.

**Limitations & ideas**
* ASR often "auto-corrects" ungrammatical speech, hiding errors – a larger Whisper model for transcripts helps the text view.
* Labels are sparse at the low end (0–1.5); such scores are over-predicted (regression to the mean) – see the per-label error table.
* Try `ENC_SIZE = "medium"`, add a second acoustic encoder (wav2vec2 / WavLM), or fine-tune an encoder end-to-end with more compute.